In [12]:
from typing import Literal , TypedDict ,Annotated , List ,operator
from langgraph.graph import StateGraph , START, END
from langchain_ollama import ChatOllama
from pydantic import Field, BaseModel
model = "qwen3.5:9b"
 

llm = ChatOllama(
    model=model,
    temperature=0
)

### STATE


In [13]:
# state
class PostState(TypedDict):
    topic: str
    post: str
    evaluation: Literal["approved", "needs_improvement"]
    feedback: str
    iteration: int
    max_iteration: int

    post_history: Annotated[List[str], operator.add]
    feedback_history: Annotated[List[str], operator.add]

In [14]:
from typing import TypedDict, Literal, Annotated, List
import operator

from langgraph.graph import StateGraph, START, END
from langchain_ollama import ChatOllama


# -----------------------------
# 1. State
# -----------------------------
class PostState(TypedDict):
    topic: str
    post: str
    evaluation: Literal["approved", "needs_improvement"]
    feedback: str
    iteration: int
    max_iteration: int

    post_history: Annotated[List[str], operator.add]
    feedback_history: Annotated[List[str], operator.add]


# -----------------------------
# 2. LLM
# -----------------------------
llm = ChatOllama(
    model="qwen3.5:9b",
    temperature=0
)


# -----------------------------
# 3. Generate Node
# -----------------------------
def generate(state: PostState):
    prompt = f"""
Write a clear and professional social media post about:

Topic: {state["topic"]}
"""

    response = llm.invoke(prompt)

    return {
        "post": response.content,
        "iteration": 1,
        "post_history": [response.content]
    }


# -----------------------------
# 4. Evaluate Node
# -----------------------------
def evaluate(state: PostState):
    prompt = f"""
            Evaluate the following social media post.

            Topic:
            {state["topic"]}

            Post:
            {state["post"]}

            Decide whether the post is good enough.

            Return exactly one of these two words:

            approved
            needs_improvement
            """

    response = llm.invoke(prompt)

    result = response.content.strip().lower()

    if "approved" in result and "needs_improvement" not in result:
        evaluation = "approved"
        feedback = "The post is acceptable."
    else:
        evaluation = "needs_improvement"

        feedback_prompt = f"""
        Explain briefly how this post can be improved.

        Topic:
        {state["topic"]}

        Post:
        {state["post"]}
        """

        feedback_response = llm.invoke(feedback_prompt)
        feedback = feedback_response.content

    return {
        "evaluation": evaluation,
        "feedback": feedback,
        "feedback_history": [feedback]
    }


# -----------------------------
# 5. Optimize Node
# -----------------------------
def optimize(state: PostState):
    prompt = f"""
            Improve the following social media post using the feedback.

            Topic:
            {state["topic"]}

            Current Post:
            {state["post"]}

            Feedback:
            {state["feedback"]}

            Return only the improved post.
            """

    response = llm.invoke(prompt)

    new_post = response.content

    return {
        "post": new_post,
        "iteration": state["iteration"] + 1,
        "post_history": [new_post]
    }


# -----------------------------
# 6. Conditional Router
# -----------------------------
def route_evaluation(state: PostState):
    if state["evaluation"] == "approved":
        return "approved"

    if state["iteration"] >= state["max_iteration"]:
        return "approved"

    return "needs_improvement"


# -----------------------------
# 7. Build Graph
# -----------------------------
graph = StateGraph(PostState)

graph.add_node("generate", generate)
graph.add_node("evaluate", evaluate)
graph.add_node("optimize", optimize)


graph.add_edge(START, "generate")

graph.add_edge("generate", "evaluate")

graph.add_conditional_edges(
    "evaluate",
    route_evaluation,
    {
        "approved": END,
        "needs_improvement": "optimize"
    }
)

graph.add_edge("optimize", "evaluate")


# -----------------------------
# 8. Compile
# -----------------------------
workflow = graph.compile()


# -----------------------------
# 9. Run
# -----------------------------
initial_state = {
    "topic": "Artificial Intelligence in industrial automation",
    "post": "",
    "evaluation": "needs_improvement",
    "feedback": "",
    "iteration": 0,
    "max_iteration": 2,
    "post_history": [],
    "feedback_history": []
}


result = workflow.invoke(initial_state)

print("Final Post:")
print(result["post"])

print("\nEvaluation:")
print(result["evaluation"])

print("\nIterations:")
print(result["iteration"])

print("\nPost History:")
for i, post in enumerate(result["post_history"], 1):
    print(f"\nVersion {i}:")
    print(post)

Final Post:
Here are three options tailored for different platforms and specific tones. You can choose the one that best fits your brand voice.

### Option 1: LinkedIn (Thought Leadership & Professional)
*Best for: Establishing authority, engaging with industry peers.*

**Headline:** The Future of Manufacturing is Intelligent. 🏭🤖

Artificial Intelligence is no longer just a buzzword; it is the backbone of modern industrial automation. By integrating AI into our production lines, we aren't just automating tasks—we are redefining efficiency and safety.

Key impacts we are seeing today:
✅ **Predictive Maintenance:** Anticipating equipment failures before they happen to minimize downtime.
✅ **Quality Control:** Computer vision detecting defects faster than the human eye.
✅ **Safety:** Autonomous systems handling hazardous environments, keeping our workforce safe.

The goal isn't to replace human expertise, but to empower it with data-driven insights. As we move toward Industry 4.0, how is 